In [0]:
storage_account = "stedukronretaildl"

silver_base = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/processed"
)

gold_base = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/processed"
)

print("Silver:", silver_base)
print("Gold:", gold_base)

In [0]:
gold_test_path = (
    "abfss://gold@stedukronretaildl.dfs.core.windows.net/test"
)

dbutils.fs.mkdirs(gold_test_path)

print("Gold storage write test successful!")

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Load Silver datasets
products = spark.read.format("delta").load(f"{silver_base}/blob/products")
brands = spark.read.format("delta").load(f"{silver_base}/blob/brands")
categories = spark.read.format("delta").load(f"{silver_base}/blob/categories")
prices = spark.read.format("delta").load(f"{silver_base}/blob/product_prices")
attributes = spark.read.format("delta").load(f"{silver_base}/blob/product_attributes")

print("All product datasets loaded successfully!")

In [0]:
products.printSchema()
brands.printSchema()
categories.printSchema()
prices.printSchema()
attributes.printSchema()

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Select the latest price for each product
price_window = Window.partitionBy("product_id").orderBy(
    col("effective_from").desc(),
    col("price_id").desc()
)

latest_prices = (
    prices
    .withColumn("row_num", row_number().over(price_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# Join all product-related datasets
dim_product = (
    products.alias("p")
    .join(brands.alias("b"), "brand_id", "left")
    .join(categories.alias("c"), "category_id", "left")
    .join(latest_prices.alias("pr"), "product_id", "left")
    .join(attributes.alias("a"), "product_id", "left")
    .select(
        col("product_id"),
        col("sku"),
        col("product_name"),
        col("brand_name"),
        col("category_name"),
        col("supplier_id"),
        col("unit_cost"),
        col("unit_price"),
        col("currency"),
        col("effective_from"),
        col("color"),
        col("size"),
        col("weight_kg")
    )
)

# Write Gold dimension
dim_product.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/dim_product")

print("dim_product created successfully!")
print("Total products:", dim_product.count())

In [0]:
display(
    spark.read.format("delta")
    .load(f"{gold_base}/dim_product")
)

In [0]:
print("Product rows:", dim_product.count())
print("Distinct products:", dim_product.select("product_id").distinct().count())

In [0]:
customers = spark.read.format("delta").load(
    f"{silver_base}/sql/customers"
)

customer_addresses = spark.read.format("delta").load(
    f"{silver_base}/sql/customer_addresses"
)

customers.printSchema()
customer_addresses.printSchema()

In [0]:
from pyspark.sql.functions import count

address_counts = (
    customer_addresses
    .groupBy("customer_id")
    .agg(count("address_id").alias("address_count"))
)

display(
    address_counts
    .groupBy("address_count")
    .count()
    .orderBy("address_count")
)

In [0]:
print("Total customers:", customers.count())
print("Total addresses:", customer_addresses.count())
print(
    "Customers with multiple addresses:",
    address_counts.filter(col("address_count") > 1).count()
)

In [0]:
from pyspark.sql.functions import col, concat_ws

dim_customer = (
    customers.alias("c")
    .join(
        customer_addresses.alias("a"),
        col("c.customer_id") == col("a.customer_id"),
        "left"
    )
    .select(
        col("c.customer_id"),
        col("c.first_name"),
        col("c.last_name"),
        concat_ws(
            " ",
            col("c.first_name"),
            col("c.last_name")
        ).alias("customer_name"),
        col("c.email"),
        col("c.loyalty_tier"),
        col("c.created_at"),
        col("a.address_id"),
        col("a.city"),
        col("a.state"),
        col("a.country"),
        col("a.postal_code")
    )
)

# Write to Gold
dim_customer.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/dim_customer")

print("dim_customer created successfully!")

In [0]:
print("Customer rows:", dim_customer.count())

print(
    "Distinct customers:",
    dim_customer.select("customer_id").distinct().count()
)

display(dim_customer)

In [0]:
# Load Silver datasets
stores = spark.read.format("delta").load(
    f"{silver_base}/blob/stores"
)

warehouses = spark.read.format("delta").load(
    f"{silver_base}/adls/warehouses"
)

stores.printSchema()
warehouses.printSchema()

In [0]:
from pyspark.sql.functions import col

dim_store = (
    stores.alias("s")
    .join(
        warehouses.alias("w"),
        col("s.warehouse_id") == col("w.warehouse_id"),
        "left"
    )
    .select(
        col("s.store_id"),
        col("s.store_name"),
        col("s.city").alias("store_city"),
        col("s.warehouse_id"),
        col("w.warehouse_name"),
        col("w.city").alias("warehouse_city")
    )
)

# Write to Gold
dim_store.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/dim_store")

print("dim_store created successfully!")

In [0]:
print("Store rows:", dim_store.count())

print(
    "Distinct stores:",
    dim_store.select("store_id").distinct().count()
)

display(dim_store)

In [0]:
orders = spark.read.format("delta").load(
    f"{silver_base}/sql/orders"
)

order_items = spark.read.format("delta").load(
    f"{silver_base}/sql/order_items"
)

orders.printSchema()
order_items.printSchema()

In [0]:
from pyspark.sql.functions import col

fact_sales = (
    order_items.alias("oi")
    .join(
        orders.alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )
    .select(
        # Transaction identifiers
        col("oi.order_item_id"),
        col("oi.order_id"),

        # Dimension keys
        col("oi.product_id"),
        col("o.customer_id"),
        col("o.store_id"),
        col("o.channel_id"),
        col("o.employee_id"),

        # Transaction details
        col("o.order_date"),
        col("o.status"),
        col("oi.promotion_id"),

        # Sales measures
        col("oi.quantity"),
        col("oi.unit_price"),
        col("oi.discount_pct"),
        col("oi.line_amount")
    )
)

# Write to Gold
fact_sales.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/fact_sales")

print("fact_sales created successfully!")

In [0]:
print("Fact sales rows:", fact_sales.count())

print(
    "Distinct order items:",
    fact_sales.select("order_item_id").distinct().count()
)

print(
    "Distinct orders:",
    fact_sales.select("order_id").distinct().count()
)

display(fact_sales)

In [0]:
remaining_tables = {
    "returns": "sql",
    "product_reviews": "api",
    "support_tickets": "api",
    "web_sessions": "api",
    "inventory_snapshots": "adls",
    "shipments": "adls",
    "delivery_events": "api"
}

for table_name, source in remaining_tables.items():
    print(f"\n{'='*60}")
    print(f"TABLE: {table_name}")
    print("="*60)

    df = spark.read.format("delta").load(
        f"{silver_base}/{source}/{table_name}"
    )

    df.printSchema()

In [0]:
from pyspark.sql.functions import (
    sum, countDistinct, round
)

sales_daily = (
    fact_sales
    .groupBy("order_date")
    .agg(
        round(sum("line_amount"), 2).alias("total_revenue"),
        sum("quantity").alias("total_units_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy("order_date")
)

sales_daily.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/sales_daily")

print("sales_daily created successfully!")
display(sales_daily)

In [0]:
returns = spark.read.format("delta").load(
    f"{silver_base}/sql/returns"
)

reviews = spark.read.format("delta").load(
    f"{silver_base}/api/product_reviews"
)

tickets = spark.read.format("delta").load(
    f"{silver_base}/api/support_tickets"
)

sessions = spark.read.format("delta").load(
    f"{silver_base}/api/web_sessions"
)

In [0]:
from pyspark.sql.functions import (
    sum, count, avg, when, round
)

# Sales metrics
customer_sales = (
    fact_sales
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("line_amount"), 2).alias("total_revenue"),
        sum("quantity").alias("total_units_purchased")
    )
)

# Return metrics
customer_returns = (
    returns
    .groupBy("customer_id")
    .agg(
        count("return_id").alias("total_returns")
    )
)

# Review metrics
customer_reviews = (
    reviews
    .groupBy("customer_id")
    .agg(
        count("review_id").alias("total_reviews"),
        round(avg("rating"), 2).alias("average_rating")
    )
)

# Support ticket metrics
customer_tickets = (
    tickets
    .groupBy("customer_id")
    .agg(
        count("ticket_id").alias("total_tickets"),
        sum(
            when(
                col("status").isin("Open", "In Progress"),
                1
            ).otherwise(0)
        ).alias("open_tickets")
    )
)

# Web session metrics
customer_sessions = (
    sessions
    .groupBy("customer_id")
    .agg(
        count("session_id").alias("total_sessions"),
        countDistinct("channel_id").alias("channels_used")
    )
)

In [0]:
customer_360 = (
    dim_customer.alias("c")
    .join(customer_sales, "customer_id", "left")
    .join(customer_returns, "customer_id", "left")
    .join(customer_reviews, "customer_id", "left")
    .join(customer_tickets, "customer_id", "left")
    .join(customer_sessions, "customer_id", "left")
    .fillna(0, subset=[
        "total_orders",
        "total_revenue",
        "total_units_purchased",
        "total_returns",
        "total_reviews",
        "total_tickets",
        "open_tickets",
        "total_sessions",
        "channels_used"
    ])
)

customer_360.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/customer_360")

print("customer_360 created successfully!")
display(customer_360)

In [0]:
from pyspark.sql.functions import when

warehouses = spark.read.format("delta").load(
    f"{silver_base}/adls/warehouses"
)

inventory = spark.read.format("delta").load(
    f"{silver_base}/adls/inventory_snapshots"
)

inventory_health = (
    inventory.alias("i")
    .join(
        dim_product.alias("p"),
        col("i.product_id") == col("p.product_id"),
        "left"
    )
    .join(
        warehouses.alias("w"),
        col("i.warehouse_id") == col("w.warehouse_id"),
        "left"
    )
    .select(
        col("i.snapshot_id"),
        col("i.snapshot_date"),
        col("i.product_id"),
        col("p.product_name"),
        col("i.warehouse_id"),
        col("w.warehouse_name"),
        col("i.quantity_on_hand"),
        col("i.reorder_level"),
        when(
            col("i.quantity_on_hand") <= col("i.reorder_level"),
            "Reorder Required"
        ).otherwise("Stock Healthy").alias("stock_status"),
        when(
            col("i.quantity_on_hand") < col("i.reorder_level"),
            col("i.reorder_level") - col("i.quantity_on_hand")
        ).otherwise(0).alias("replenishment_quantity")
    )
)

inventory_health.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/inventory_health")

print("inventory_health created successfully!")
display(inventory_health)

In [0]:
from pyspark.sql.functions import (
    max, collect_set, count
)

shipments = spark.read.format("delta").load(
    f"{silver_base}/adls/shipments"
)

delivery_events = spark.read.format("delta").load(
    f"{silver_base}/api/delivery_events"
)

shipment_metrics = (
    shipments
    .groupBy("order_id")
    .agg(
        count("shipment_id").alias("total_shipments"),
        max("shipment_date").alias("latest_shipment_date"),
        collect_set("status").alias("shipment_statuses"),
        collect_set("warehouse_id").alias("warehouse_ids")
    )
)

In [0]:
delivery_metrics = (
    delivery_events
    .groupBy("order_id")
    .agg(
        count("delivery_event_id").alias("total_delivery_events"),
        max("event_time").alias("latest_delivery_event"),
        collect_set("event_type").alias("delivery_event_types"),
        collect_set("carrier").alias("carriers")
    )
)

In [0]:
fulfillment_summary = (
    orders.alias("o")
    .join(shipment_metrics, "order_id", "left")
    .join(delivery_metrics, "order_id", "left")
    .fillna(0, subset=[
        "total_shipments",
        "total_delivery_events"
    ])
)

fulfillment_summary.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/fulfillment_summary")

print("fulfillment_summary created successfully!")
display(fulfillment_summary)

In [0]:
gold_tables = [
    "dim_product",
    "dim_customer",
    "dim_store",
    "fact_sales",
    "sales_daily",
    "customer_360",
    "inventory_health",
    "fulfillment_summary"
]

for table_name in gold_tables:

    df = spark.read.format("delta").load(
        f"{gold_base}/{table_name}"
    )

    print(
        f"{table_name:25} | "
        f"Rows: {df.count():6}"
    )